In [2]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec,Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [3]:
load_dotenv()

True

In [4]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.7
)

In [5]:
import os
pc=Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [6]:
index_name = "practice" 

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [7]:
file_path = "D:\ProdRAG\prodRAG\Blockchain_Course_Proposal.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [11]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview",output_dimensionality=384)

In [12]:
vectorstore = PineconeVectorStore.from_documents(
    texts,
    embedder,
    index_name=index_name
)

In [13]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
query = "What is the module4?"
results = retriever.invoke(query)

In [14]:
print(results)

[Document(id='192fd876-4d75-4c2c-8dc1-962ca41e6260', metadata={'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, page_content='blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \nand training data provenance.'), Document(id='92ca04fd-5eb7-494a-a122-0a818fc78fdd', metadata={'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30',

In [15]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [16]:
rag_prompt = PromptTemplate.from_template(
   template= """You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [17]:
rag_chain=RunnableParallel({
    "question" :RunnablePassthrough(),
    "context" :retriever| RunnableLambda(format_docs)
}
)

In [18]:
rag_chain.invoke("What is 4th module?")

{'question': 'What is 4th module?',
 'context': 'Network access will be required to interact with test blockchain environments such as\n\nModule 7: Cybersecurity and Blockchain — Role of blockchain in identity management,\n\napplication development. \n8. Resource Requirements \nThe course will require mid-range to high-end computing systems with at least 8 GB RAM \nand GPU-enabled setups for blockchain simulation. \nSoftware requirements include Truffle Suite, Metamask, Solidity Compiler, Python, and \nNode.js. \nLibraries such as Web3.py, Web3.js, Pandas, and Scikit-learn will be used for AI and \nblockchain integration. \nNetwork access will be required to interact with test blockchain environments such as'}

In [19]:
parser=StrOutputParser()

In [20]:
output=rag_chain | rag_prompt | llm | parser

In [21]:
output.invoke("What is 4th module?")

'The 4th module is: Smart Contracts and Ethereum — Writing and deploying smart contracts using Solidity and Remix IDE.'